Objective: Analyze foldseek clustering data of CATH and ECOD domains. Determine if members of the same cluster have the same classifications.

In [1]:
#import libraries
import pandas as pd
import requests
import os


In [2]:
#define project path
project_path = "/Users/karenzhu/Desktop/Projects/MAIN_structure_project/work/SP001_050/"
CATH_cluster_path = os.path.join(project_path, "SP020/CATH/CATH_easy_cluster_cluster.tsv")
CATH_annotations_path = os.path.join(project_path, "SP017/CATH_files/CATH_S20_domain_annotations.csv")

# CATH Data Wrangling and Exploration
 * calculate the size of each cluster by clustering by the representative
 * for each representative and member, get its corresponding cath code and annotation
 * Determine if the cath codes for the rep and members match. Assign 1 if match and 0 if mismatch
 * for each cluster, calculate the fraction of CATH_code matches between the rep and members.
 * for the mismatched cath codes, output them in a file with their cath code and annotations


In [3]:
#read in files
CATH_clusters_df = pd.read_csv(CATH_cluster_path, sep='\t', header=None, names = ['Representative', 'Member'])

CATH_annotations_df = pd.read_csv(CATH_annotations_path)
CATH_annotations_df = CATH_annotations_df[['Domain_name', 'CATH_code', 'CATH_class', 'CATH_architecture','CATH_topology','CATH_homologous_fam' ]]

In [5]:
CATH_clusters_df[CATH_clusters_df['Representative'] == CATH_clusters_df['Member']]

,Representative,Member
0,132lA00,132lA00
1,153lA00,153lA00
2,16pkA02,16pkA02
3,16vpA00,16vpA00
4,1914A00,1914A00
...,...,...
14937,2hjmA01,2hjmA01
14938,2hjnA00,2hjnA00
14939,2hjqA01,2hjqA01
14940,2hjqA02,2hjqA02


In [43]:
#get cath code for the rep and member
CATH_clusters_df = CATH_clusters_df.merge(
    CATH_annotations_df[['Domain_name', 'CATH_code']], 
    left_on='Representative', 
    right_on='Domain_name', 
    how='left'
)

CATH_clusters_df = CATH_clusters_df.drop(columns=['Domain_name'])
CATH_clusters_df = CATH_clusters_df.rename(columns={'CATH_code': 'Rep_CATH_code'})

CATH_clusters_df = CATH_clusters_df.merge(
    CATH_annotations_df[['Domain_name', 'CATH_code']], 
    left_on='Member', 
    right_on='Domain_name', 
    how='left'
)

CATH_clusters_df = CATH_clusters_df.drop(columns=['Domain_name'])
CATH_clusters_df = CATH_clusters_df.rename(columns={'CATH_code': 'Member_CATH_code'})


In [44]:
#see if the two cath code match for representative and member. assign 1 if match and 0 if mismatch
CATH_clusters_df['Code_match_status'] = CATH_clusters_df['Rep_CATH_code'] == CATH_clusters_df['Member_CATH_code']
CATH_clusters_df['Code_match_status'] = CATH_clusters_df['Code_match_status'].astype(int)

In [45]:
#determine how many entries where the cath code for rep and member doesn't match
print("Number of mismatched classifications: ",CATH_clusters_df[CATH_clusters_df['Code_match_status']==0].shape[0])

Number of mismatched classifications:  670


In [46]:
#for each representative/cluster, determine the fraction of matched members to its rep
CATH_calc_matches_df = CATH_clusters_df.groupby('Representative').agg(
    Cluster_matches_num=('Code_match_status', 'sum'),
    Cluster_size=('Code_match_status', 'size')
).reset_index()

CATH_calc_matches_df['Match_frac'] = CATH_calc_matches_df['Cluster_matches_num']/CATH_calc_matches_df['Cluster_size']
CATH_calc_matches_df['Match_frac'] = CATH_calc_matches_df['Match_frac'].round(decimals=2)

CATH_calc_matches_df.head()

,Representative,Cluster_matches_num,Cluster_size,Match_frac
0,132lA00,1,1,1.0
1,153lA00,1,1,1.0
2,16pkA02,1,1,1.0
3,16vpA00,1,1,1.0
4,1914A00,1,1,1.0


In [48]:
#for clusters that contain mismatched cath codes, get the members in the cluster, cath codes, and the cath code annotations


#get annotations of clusters/reps  
CATH_calc_matches_df = CATH_calc_matches_df.merge(CATH_clusters_df, on='Representative', how='left')

CATH_calc_matches_df = CATH_calc_matches_df.merge(
    CATH_annotations_df[['Domain_name', 'CATH_class', 'CATH_architecture','CATH_topology','CATH_homologous_fam']], 
    left_on='Representative', 
    right_on = 'Domain_name', 
    how='left')

CATH_calc_matches_df = CATH_calc_matches_df.drop(columns=['Domain_name'])

columns_to_prefix = ['CATH_class', 'CATH_architecture','CATH_topology','CATH_homologous_fam' ]

# Add prefix to specific columns
CATH_calc_matches_df  = CATH_calc_matches_df.rename(columns={col: 'Rep_' + col for col in columns_to_prefix})

CATH_calc_matches_df = CATH_calc_matches_df.merge(CATH_annotations_df[['Domain_name','CATH_class', 'CATH_architecture','CATH_topology','CATH_homologous_fam' ]], left_on='Member', right_on = 'Domain_name', how='left')
CATH_calc_matches_df = CATH_calc_matches_df.drop(columns=['Domain_name'])
CATH_calc_matches_df  = CATH_calc_matches_df.rename(columns={col: 'Member_' + col for col in columns_to_prefix})

In [49]:
#determine the reps with <1 match_frac
CATH_mismatches_df = CATH_calc_matches_df[CATH_calc_matches_df['Match_frac'] < 1]

print("Number of clusters containing mismatched CATH codes between the rep and members: ", len(CATH_mismatches_df['Representative'].unique()))

Number of clusters containing mismatched CATH codes between the rep and members:  299


In [50]:
#determine if any groups/rep have the same cath code. OUTPUT THIS AS CSV???
reps_per_cath_code_df = CATH_clusters_df.groupby('Rep_CATH_code').agg(
    Unique_Representatives_Count=('Representative', 'nunique'),
    Unique_Representatives=('Representative', lambda x: ', '.join(x.unique()))
    ).reset_index()

reps_per_cath_code_df.columns = ['Rep_CATH_code', 'Unique_Representatives_Count', 'Representatives']
dupe_reps_per_cath_code_df = reps_per_cath_code_df[reps_per_cath_code_df['Unique_Representatives_Count']>1].sort_values(by = 'Unique_Representatives_Count', ascending=False) 
dupe_reps_per_cath_code_df  = dupe_reps_per_cath_code_df.reset_index(drop=True)
display(dupe_reps_per_cath_code_df.head())

print("Number of CATH codes with more than 1 representative: ", dupe_reps_per_cath_code_df.shape[0])
print("Number of representatives that share the CATH codes: ", dupe_reps_per_cath_code_df['Unique_Representatives_Count'].sum())
print("Number of CATH codes with only 1 representatives: ", reps_per_cath_code_df[reps_per_cath_code_df['Unique_Representatives_Count']==1].shape[0])

,Rep_CATH_code,Unique_Representatives_Count,Representatives
0,1.10.10.10,126,"1fznD00, 1g4dA00, 3go5A04, 3gw6D02, 3k0lA00, 3..."
1,3.40.50.300,90,"1e2jA00, 1e3mA05, 1g5rA00, 1g7rA01, 1g8fA03, 3..."
2,2.60.40.10,58,"3gm8A04, 3gzkA01, 3i84B00, 3lsoA02, 3mgaA01, 1..."
3,2.40.50.140,55,"1g29102, 1gd7A00, 3irbA02, 3nbhB00, 3a5zD02, 1..."
4,1.25.40.10,50,"3jysA02, 3k9iA01, 3kaeA00, 3ly7A02, 3mekA05, 3..."


Number of CATH codes with more than 1 representative:  1007
Number of representatives that share the CATH codes:  4685
Number of CATH codes with only 1 representatives:  4273


In [51]:
#output files to csv

CATH_calc_matches_df.to_csv("CATH_foldseek_cluster_annotations.csv", index=False)

CATH_mismatches_df.to_csv("CATH_foldseek_rep_vs_member_mismatches.csv", index=False)

dupe_reps_per_cath_code_df.to_csv("CATH_foldseek_multiple_reps_per_CATH_code.csv", index=False)

# Extract a few clusters with the same CATH codes
 * Look at these structures in pymol
 * Notes: Members with in different clusters that have the same CATH code have similar looking structures and have some overlap alignment. Some align more than others.

In [ ]:
#get the clusters with the same CATH_code and visualize in pymol

#cath code = 6.20.50.180. This has 2 clusters
rep_6_20_50_180 = ['2k4xA01', '2xzm901']
#display(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_6_20_50_180)])
members_6_20_50_180 = list(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_6_20_50_180)]['Member'])

#cath code = 3.40.50.1360. This has 3 clusters
rep_3_40_50_1360 = ['3hn6F00','4nmlA01', '2o0mA00' ]
#display(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_40_50_1360)])
members_3_40_50_1360 = list(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_40_50_1360)]['Member'])

#cath code = 3.30.230.10. This has 10 members.
rep_3_30_230_10 = ['1pkpA02', '1pvgA02', '4fw9A03', '1b63A02', '2xexA04', '5x8ri00', '1mg7A02', '1mx0A03', '1n0uA06', '2hfsA01']
#display(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_30_230_10)])
members_3_30_230_10 = list(CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_30_230_10)]['Member'])

In [98]:
#output selected cath codes with multiple clusters into a csv file
cluster_same_CATH_code_dfs= [
    CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_6_20_50_180)],
    CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_40_50_1360)],
    CATH_clusters_df[CATH_clusters_df['Representative'].isin(rep_3_30_230_10)]
    
]
CATH_selected_clusters_for_pymol = pd.concat(cluster_same_CATH_code_dfs, axis=0)

CATH_selected_clusters_for_pymol.to_csv("CATH_selected_clusters_for_pymol.csv", index=False)

#download the selected CATH domains to look at pymol

member_list_3 = [members_6_20_50_180] #, members_3_40_50_1360, members_3_30_230_10  ]

for lst in member_list_3:
    for CATH_code in lst:
        response = requests.get(f"http://www.cathdb.info/version/v4_3_0/api/rest/id/{CATH_code}.pdb")
        if response.status_code == 200:
            with open(f"{CATH_code}.pdb", "wb") as file:
                file.write(response.content)
        else:
            print(f"Failed to download {CATH_code}. Status code: {response.status_code}")

# Create nonredundant CATH domain dataset

In [13]:
CATH_reps_only_df = CATH_clusters_df[CATH_clusters_df['Representative'] == CATH_clusters_df['Member']]

CATH_reps_only_df = CATH_reps_only_df.drop("Member", axis = 'columns')

cluster_rep_file_name = os.path.join(project_path, 'SP020/CATH/CATH_foldseek_cluster_reps.txt')
CATH_reps_only_df.to_csv(cluster_rep_file_name, header=False, index = False)